# View Factor Computation

Layer 1 of 3. Computes view factors for all spacecraft surfaces and saves to `outputs/viewfactors.npz`.

**Requires:** `outputs/spacecraft.json` (from `run_cubesat_geometry.ipynb`)

**Runtime:** ~5–10 min depending on quadrature settings.

In [1]:
from pathlib import Path
from datetime import datetime
import json, math
import numpy as np
import matplotlib.pyplot as plt

from geometry import Orbit, RealizedGeometry, SlewModeSwitch, SunTracking, TargetTracking
from viewfactor import surface_loading_propagate, save_profiles

# -- Paths ------------------------------------------------------------------
GEOMETRY_FILE   = Path('outputs/spacecraft.json')
BODY_ROLES_JSON = Path('outputs/body_roles.json')
VF_NPZ          = Path('outputs/viewfactors.npz')

# -- Orbit ------------------------------------------------------------------
A_KM      = 6378 + 450
INC_DEG   = 51.6
OMEGA_DEG = 45.0
EPOCH     = datetime(2026, 6, 22, 12, 0, 0)

# -- Attitude ---------------------------------------------------------------
TARGET_RA_DEG  = 266.4168
TARGET_DEC_DEG = -29.0078
SLEW_DEG_S     = 1.0

# -- View factor quadrature -------------------------------------------------
N_ORBIT   = 360    # orbit samples
N_MU      = 24    # Earth-disk polar samples
N_AZ      = 72    # Earth-disk azimuth samples
HEMI_N_AZ = 73    # hemisphere azimuth samples (static VFs)
HEMI_N_EL = 33    # hemisphere elevation samples (static VFs)

# -- Surface selection ------------------------------------------------------
# Add tag strings to exclude surface groups.  Empty = compute everything.
# Example: SKIP_TAGS = {'solar_panel_back'} to skip panel back faces.
SKIP_TAGS = set()

In [2]:
# -- Load geometry ---------------------------------------------------------
realized   = RealizedGeometry.from_json(GEOMETRY_FILE)
body_roles = json.loads(BODY_ROLES_JSON.read_text(encoding='utf-8'))

surfaces = [s for s in realized.surfaces
            if not any(t in s.tags for t in SKIP_TAGS)]

print(f'{len(surfaces)} surfaces selected:')
for s in surfaces:
    tags = ', '.join(sorted(s.tags))
    print(f'  {s.name:<32s}  normal={tuple(s.normal.round(0).astype(int))}  {tags}')

14 surfaces selected:
  bus_+X                            normal=(0, 1, 0)  +X, bus
  bus_-X                            normal=(0, -1, 0)  -X, bus
  bus_+Y                            normal=(0, 0, 1)  +Y, bus
  bus_-Y                            normal=(0, 0, -1)  -Y, bus
  bus_+Z                            normal=(1, 0, 0)  +Z, bus
  bus_-Z                            normal=(-1, 0, 0)  -Z, bus
  wing_port_inner                   normal=(0, 0, 1)  deployable, inner, port, solar_array, solar_panel
  wing_port_inner_back              normal=(0, 0, -1)  deployable, inner, port, solar_array, solar_panel_back
  wing_port_outer                   normal=(0, 0, 1)  deployable, outer, port, solar_array, solar_panel
  wing_port_outer_back              normal=(0, 0, -1)  deployable, outer, port, solar_array, solar_panel_back
  wing_starboard_inner              normal=(0, 0, 1)  deployable, inner, solar_array, solar_panel, starboard
  wing_starboard_inner_back         normal=(0, 0, -1)  deployable,

In [3]:
# -- Orbit + attitude law -------------------------------------------------
orbit = Orbit.from_epoch(
    a=A_KM * 1e3,
    i=math.radians(INC_DEG),
    omega=math.radians(OMEGA_DEG),
    epoch=EPOCH,
)
law = SlewModeSwitch(
    TargetTracking(math.radians(TARGET_RA_DEG), math.radians(TARGET_DEC_DEG)),
    SunTracking(),
    slew_rate_deg_s=SLEW_DEG_S,
)
print(f'Orbit period: {orbit.period:.1f} s  ({orbit.period / 60:.1f} min)')

Orbit period: 5615.0 s  (93.6 min)


In [ ]:
# -- View factor propagation -----------------------------------------------
vf_kwargs = dict(
    n=N_ORBIT, n_mu=N_MU, n_az=N_AZ,
    hemi_n_az=HEMI_N_AZ, hemi_n_el=HEMI_N_EL,
)

profiles = []
for s in surfaces:
    print(f'  {s.name}...', end=' ', flush=True)
    profiles.append(surface_loading_propagate(realized, s.name, orbit, law, **vf_kwargs))
    print(f'n={profiles[-1].u.size}')
print('Done.')

  bus_+X... n=410
  bus_-X... 

In [ ]:
# -- Save -----------------------------------------------------------------
VF_NPZ.parent.mkdir(exist_ok=True)
save_profiles(profiles, VF_NPZ, orbit_period=orbit.period)
print(f'Saved {len(profiles)} profiles -> {VF_NPZ}')

In [ ]:
# -- Validation: radiator Earth view factor --------------------------------
by_name = {p.surface_name: p for p in profiles}
u_deg   = np.degrees(profiles[0].u)
eclipse = profiles[0].eclipse

radiators = [
    (by_name[body_roles[ax]], ax)
    for ax in ('+Y', '-Y')
    if body_roles.get(ax) in by_name
]

fig, ax = plt.subplots(figsize=(10, 3))
ax.fill_between(u_deg, 0, 1, where=eclipse, alpha=0.12, color='steelblue', label='eclipse')
for profile, label in radiators:
    ax.plot(u_deg, profile.average_earth_view(), lw=1.5, label=profile.surface_name)
ax.set_xlabel('u [deg]')
ax.set_ylabel('Earth view factor')
ax.set_title('Radiator Earth view factor — validation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()